# Пробное обучение RTMDet-Ins-Tiny для Skalyk

Отдельная экспериментальная модель сегментации зацепок. Она обучается на Climbing Hold Detection v26 в формате COCO и использует два класса: `hold` и `volume`. В экспорте Roboflow названия `hold` и `holds` приводятся к одному классу `hold`.

Для Kaggle используется отдельное окружение Python 3.10 + PyTorch 2.1.0 + CUDA 12.1 + готовый MMCV 2.1.0 wheel. Это обходит сборку MMCV под стандартный Python 3.13 Kaggle. Если совместимое готовое колесо не загрузится, установка завершится сразу с понятной ошибкой, не пытаясь долго собирать библиотеку из исходников.

Включите Internet и бесплатный GPU в настройках Kaggle. Сначала ноутбук выполнит короткий smoke-прогон на 24 изображениях, затем запустит полноценное обучение на одном GPU с batch size 2. Старт/финиш в датасете не размечены.


In [ ]:
%pip -q install uv roboflow

from pathlib import Path
import shutil
import subprocess
import sys

UV = shutil.which('uv')
if UV is None:
    raise RuntimeError('Не удалось установить uv в окружение Kaggle.')

WORK = Path('/kaggle/working')
ENV_DIR = WORK / 'rtmdet-python310'
ENV_PY = ENV_DIR / 'bin' / 'python'

# Изолированная версия Python с готовыми совместимыми CUDA-пакетами.
subprocess.run([UV, 'python', 'install', '3.10'], check=True)
if ENV_PY.exists():
    print('Переиспользую готовое Python 3.10 окружение:', ENV_DIR)
else:
    # --clear удалит только неполную папку окружения, если она осталась после прерванного создания.
    subprocess.run([UV, 'venv', '--clear', '--python', '3.10', str(ENV_DIR)], check=True)

# PyTorch с CUDA 12.1 работает на Kaggle T4 и не меняет PyTorch основного ядра.
subprocess.run([
    UV, 'pip', 'install', '--python', str(ENV_PY),
    '--index-url', 'https://download.pytorch.org/whl/cu121',
    'torch==2.1.0', 'torchvision==0.16.0'
], check=True)

# MMCV ставится только готовым wheel: компиляция на Kaggle здесь не запускается.
subprocess.run([
    UV, 'pip', 'install', '--python', str(ENV_PY),
    '--find-links', 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html',
    '--only-binary', 'mmcv',
    'numpy<2', 'setuptools==80.9.0', 'mmcv==2.1.0', 'mmengine==0.10.7', 'mmdet==3.3.0'
], check=True)

# Проверяем именно CUDA-операцию MMCV, нужную RTMDet, прежде чем скачивать набор.
preflight = r'''import torch, mmcv, mmengine, mmdet, setuptools
from mmcv.ops import nms
assert torch.cuda.is_available(), "CUDA недоступна в отдельном Python 3.10 окружении"
boxes = torch.tensor([[0., 0., 10., 10.]], device="cuda")
scores = torch.tensor([0.9], device="cuda")
_, keep = nms(boxes, scores, 0.5)
assert keep.numel() == 1, "CUDA NMS MMCV не прошёл проверку"
print("Python / Torch / MMCV / MMDetection:", __import__("sys").version.split()[0], torch.__version__, mmcv.__version__, mmdet.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("MMCV CUDA NMS: OK; setuptools:", setuptools.__version__)'''
subprocess.run([str(ENV_PY), '-c', preflight], check=True)

# Официальная конфигурация RTMDet-Ins-Tiny из версии MMDetection 3.3.0.
REPO = WORK / 'mmdetection'
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'v3.3.0',
        'https://github.com/open-mmlab/mmdetection.git', str(REPO)
    ], check=True)
print('Среда готова:', ENV_DIR)


## 1. Скачать набор зацепок

Введите ключ Roboflow в скрытое поле. Он не выводится и не записывается в ноутбук. Если набор уже скачан в текущей Kaggle-сессии, этот шаг пропускается.


In [ ]:
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
import json

DATASET_DIR = Path('/kaggle/working/climbing-holds-v26')
train_json = DATASET_DIR / 'train' / '_annotations.coco.json'
valid_json = DATASET_DIR / 'valid' / '_annotations.coco.json'

if not train_json.exists() or not valid_json.exists():
    api_key = getpass('Roboflow API key (ввод скрыт): ')
    project = Roboflow(api_key=api_key).workspace('climbtag').project('climbing-hold-detection-7uehq')
    dataset = project.version(26).download('coco', location=str(DATASET_DIR))
    DATASET_DIR = Path(dataset.location)
    del api_key
    train_json = DATASET_DIR / 'train' / '_annotations.coco.json'
    valid_json = DATASET_DIR / 'valid' / '_annotations.coco.json'

if not train_json.exists() or not valid_json.exists():
    raise FileNotFoundError('В скачанном COCO-наборе не найдены train/valid аннотации.')

CLASS_ALIASES = {'hold': 'hold', 'holds': 'hold', 'volume': 'volume'}
CANONICAL_CLASSES = ('hold', 'volume')
CANONICAL_IDS = {'hold': 1, 'volume': 2}

def normalize_coco(path):
    data = json.loads(path.read_text())
    original_to_name = {}
    for category in data['categories']:
        raw_name = str(category['name']).strip().casefold()
        if raw_name not in CLASS_ALIASES:
            raise ValueError(f'Неожиданный класс {category["name"]!r} в {path.name}')
        original_to_name[category['id']] = CLASS_ALIASES[raw_name]
    if set(original_to_name.values()) != set(CANONICAL_CLASSES):
        raise ValueError(f'В {path.name} ожидались классы hold/holds и volume; получено: {data["categories"]}')
    for annotation in data['annotations']:
        old_id = annotation['category_id']
        if old_id not in original_to_name:
            raise ValueError(f'Аннотация с неизвестным category_id={old_id} в {path.name}')
        annotation['category_id'] = CANONICAL_IDS[original_to_name[old_id]]
    data['categories'] = [
        {'id': CANONICAL_IDS[name], 'name': name} for name in CANONICAL_CLASSES
    ]
    path.write_text(json.dumps(data, ensure_ascii=False))
    return data

train_data = normalize_coco(train_json)
for split in ('valid', 'test'):
    split_json = DATASET_DIR / split / '_annotations.coco.json'
    if split_json.exists():
        normalize_coco(split_json)

classes = tuple(item['name'] for item in train_data['categories'])
segmented = sum(bool(item.get('segmentation')) for item in train_data['annotations'])
if not segmented:
    raise ValueError('В COCO-аннотациях не найдены полигоны сегментации.')

print('Папка данных:', DATASET_DIR)
print('Классы после объединения hold/holds:', classes)
print('Изображений для обучения:', len(train_data['images']))
print('Полигональных масок:', segmented)


## 2. Создать конфигурацию и проверить коротким запуском

Smoke-прогон использует 24 тренировочных изображения и весь validation split. Он нужен, чтобы проверить чтение полигонов, CUDA и обучение до полного запуска.


In [ ]:
from pathlib import Path
import os
import subprocess
import textwrap

TRAIN_SCRIPT = Path('/kaggle/working/train_rtmdet_climbing.py')
TRAIN_SCRIPT.write_text(textwrap.dedent(r'''    import copy
    import json
    import math
    import os
    os.environ['MPLBACKEND'] = 'Agg'  # дочернее окружение без Jupyter backend Kaggle
    from pathlib import Path

    import torch
    from mmengine import Config
    from mmengine.runner import Runner
    from mmdet.utils import register_all_modules

    register_all_modules(init_default_scope=True)
    torch.cuda.set_device(0)

    data_root = Path(os.environ['DATASET_DIR'])
    repo = Path('/kaggle/working/mmdetection')
    mode = os.environ['RUN_MODE']
    max_epochs = int(os.environ['MAX_EPOCHS'])
    smoke = mode == 'smoke'
    batch_size = 2

    train_ann = data_root / 'train' / '_annotations.coco.json'
    valid_ann = data_root / 'valid' / '_annotations.coco.json'
    if not train_ann.exists() or not valid_ann.exists():
        raise FileNotFoundError('Не найдены COCO train/valid annotations.')

    coco = json.loads(train_ann.read_text())
    classes = tuple(cat['name'] for cat in sorted(coco['categories'], key=lambda x: x['id']))
    if {name.lower() for name in classes} != {'hold', 'volume'}:
        raise ValueError(f'Неверный список классов: {classes}')
    metainfo = dict(classes=classes, palette=[(255, 80, 40), (40, 180, 255)])

    config_path = repo / 'configs/rtmdet/rtmdet-ins_tiny_8xb32-300e_coco.py'
    cfg = Config.fromfile(str(config_path))
    cfg.model.bbox_head.num_classes = len(classes)

    def point_dataset(loader_name, split):
        loader = cfg[loader_name]
        dataset_cfg = loader.dataset
        dataset_cfg.data_root = str(data_root) + '/'
        dataset_cfg.ann_file = f'{split}/_annotations.coco.json'
        dataset_cfg.data_prefix = dict(img=f'{split}/')
        dataset_cfg.metainfo = metainfo
        loader.batch_size = 1 if loader_name != 'train_dataloader' else batch_size
        loader.num_workers = 2
        loader.persistent_workers = False

    point_dataset('train_dataloader', 'train')
    point_dataset('val_dataloader', 'valid')
    test_split = 'test' if (data_root / 'test' / '_annotations.coco.json').exists() else 'valid'
    cfg.test_dataloader = copy.deepcopy(cfg.val_dataloader)
    cfg.test_dataloader.dataset.ann_file = f'{test_split}/_annotations.coco.json'
    cfg.test_dataloader.dataset.data_prefix = dict(img=f'{test_split}/')
    cfg.val_evaluator.ann_file = str(valid_ann)
    cfg.test_evaluator.ann_file = str(data_root / test_split / '_annotations.coco.json')

    # Ограничиваем первую проверку, оставляя полную модель и CUDA-операции без изменений.
    if smoke:
        cfg.train_dataloader.dataset.indices = min(24, len(coco['images']))

    cfg.train_cfg.max_epochs = max_epochs
    cfg.train_cfg.val_interval = 1
    cfg.train_cfg.dynamic_intervals = [] if smoke else [(max(1, max_epochs - 5), 1)]
    cfg.train_dataloader.batch_size = batch_size
    cfg.optim_wrapper.type = 'AmpOptimWrapper'
    cfg.optim_wrapper.loss_scale = 'dynamic'
    cfg.optim_wrapper.optimizer.lr = 0.004 * batch_size / 256
    cfg.default_hooks.logger.interval = 10 if smoke else 50
    cfg.default_hooks.checkpoint.interval = 1
    cfg.default_hooks.checkpoint.max_keep_ckpts = 2
    cfg.default_hooks.checkpoint.save_best = 'coco/segm_mAP'
    cfg.default_hooks.checkpoint.rule = 'greater'
    cfg.log_processor.window_size = 10
    cfg.randomness = dict(seed=42, deterministic=False)

    # Для короткого smoke-прогона отключаем смену pipeline в конце обучения.
    custom_hooks = []
    for hook in cfg.custom_hooks:
        if hook.type == 'PipelineSwitchHook':
            if smoke:
                continue
            hook.switch_epoch = max(1, max_epochs - 5)
        custom_hooks.append(hook)
    cfg.custom_hooks = custom_hooks

    # Масштабируем линейный warmup к размеру набора, а не оставляем 1000 итераций для всех случаев.
    approx_iters = max(1, math.ceil(len(coco['images']) / batch_size))
    cfg.param_scheduler[0].end = min(1000, max(10, approx_iters * 2))
    cfg.param_scheduler[1].begin = max_epochs // 2
    cfg.param_scheduler[1].end = max_epochs
    cfg.param_scheduler[1].T_max = max(1, max_epochs - max_epochs // 2)

    work_dir = Path('/kaggle/working/rtmdet-ins-tiny-climbing') / mode
    cfg.work_dir = str(work_dir)
    cfg.launcher = 'none'
    work_dir.mkdir(parents=True, exist_ok=True)

    print(f'Режим: {mode}; эпох: {max_epochs}; train batch: {batch_size}; GPU: {torch.cuda.get_device_name(0)}')
    print(f'Классы: {classes}; train images: {len(coco["images"])}; validation: {valid_ann}')
    runner = Runner.from_cfg(cfg)
    runner.train()
    best = sorted(work_dir.glob('best*.pth'))
    print('Рабочая папка:', work_dir)
    print('Лучшие checkpoint-файлы:', [p.name for p in best] or 'best checkpoint пока не создан')
'''))


def run_training(mode, epochs):
    child_env = os.environ.copy()
    child_env['DATASET_DIR'] = str(DATASET_DIR)
    child_env['RUN_MODE'] = mode
    child_env['MAX_EPOCHS'] = str(epochs)
    child_env['CUDA_VISIBLE_DEVICES'] = '0'
    child_env['MPLBACKEND'] = 'Agg'
    python310 = Path('/kaggle/working/rtmdet-python310/bin/python')
    subprocess.run([UV, 'pip', 'install', '--python', str(python310), 'setuptools==80.9.0'], check=True)
    subprocess.run([str(python310), '-c', 'import sys, setuptools; print("Training Python:", sys.executable); print("setuptools:", setuptools.__version__, setuptools.__file__)'], check=True)
    subprocess.run([str(python310), str(TRAIN_SCRIPT)], env=child_env, check=True)

run_training('smoke', 1)


## 3. Полноценное обучение

Если smoke-прогон завершился и вывел CUDA-метрику сегментации, запустите эту ячейку. Начальное значение — 30 эпох и batch size 2 на одной T4. Весовая точка с лучшим `coco/segm_mAP` сохраняется отдельно от текущей.

Если Kaggle сообщает нехватку памяти, измените `batch_size = 2` в скрипте выше на `1`, затем повторно выполните ячейки создания скрипта и запуска обучения.


In [ ]:
run_training('full', 30)

from pathlib import Path
output_dir = Path('/kaggle/working/rtmdet-ins-tiny-climbing/full')
print('Модель и логи находятся в:', output_dir)
for checkpoint in sorted(output_dir.glob('*.pth')):
    print(checkpoint.name, f'{checkpoint.stat().st_size / (1024 * 1024):.1f} MiB')


## Что забрать из Kaggle

В панели Output скачайте лучший checkpoint из `/kaggle/working/rtmdet-ins-tiny-climbing/full/` и сохраните значения `coco/segm_mAP`, `coco/bbox_mAP` из журнала. PyTorch checkpoint пока не является файлом модели для браузера. После сравнения качества отдельно проверим экспорт RTMDet в ONNX и подключение его выходных масок в Skalyk.

Данные Roboflow Climbing Hold Detection v26 распространяются по CC BY 4.0: при публикации обученной модели укажите автора climbtag и ссылку на набор.
